In [ ]:
%pylab inline
plt.style.use("bmh")

In [ ]:
import pathlib
import pandas as pd

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
from tqdm import tqdm
tqdm.pandas()

In [ ]:
RS = 7345

# ???????? ??????

In [ ]:
DATA_DIR = pathlib.Path("data")

In [ ]:
%%time

train, test = [
    pd.read_csv(DATA_DIR.joinpath(filename), index_col="pair_id")
    for filename in (
        "train.csv",
        "test.csv",
    ) 
]

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.info()

In [ ]:
test.info()

# ?????????? ???????

In [ ]:
train.is_duplicate.value_counts()

In [ ]:
train[train.is_duplicate==1].sample(15)

In [ ]:
train[train.is_duplicate==0].sample(15)

# ??????? ??????

In [ ]:
import pycountry
import re

In [ ]:
%%time

for dataset in (train, test):
    for column in ("name_1", 'name_2'):
        dataset[column] = dataset[column].str.lower()

In [ ]:
train[train.name_1.str.contains("gmbh")]

In [ ]:
def multi_str_replace(strings, debug=True):
    re_str = r'\b(?:' + '|'.join(
        [re.escape(s) for s in strings]
    ) + r')(?!\S)'
    if debug:
        print(re_str)
    return re.compile(re_str, re.UNICODE)

In [ ]:
legal_entities = ["ltd.", "co.", "inc.", "b.v.", "s.c.r.l.", "gmbh", "pvt."]
legal_re = multi_str_replace([rf"{entity}" for entity in legal_entities])

t = r'sibur    gmbh inc. gmbh inc pvgmbh  b.v. bova inc.'
t = legal_re.sub('<DEL>', t)
t

In [ ]:
%%time

for dataset in (train, test):
    dataset.replace(legal_re, "$1 $3", inplace=True)

In [ ]:
%%time

for dataset in (train, test):
    dataset.replace(re.compile(r"\s+\(.*\)"), "", inplace=True)

In [ ]:
import pycountry
countries = [country.name.lower() for country in pycountry.countries]

In [ ]:
countries_re = multi_str_replace(countries)

In [ ]:
%%time

for dataset in train, test :
    dataset.replace(countries_re, "", inplace=True)

In [ ]:
%%time

for dataset in (train, test):
    dataset.replace(re.compile(r"\s+"), " ", inplace=True)
    dataset.replace(re.compile(r"[^\w\s]"), "", inplace=True)

In [ ]:
train.sample(25)

In [ ]:
test.sample(25)

# ??????? ??????? ???????? ?????

In [ ]:
from strsimpy.levenshtein import Levenshtein
from strsimpy.normalized_levenshtein import NormalizedLevenshtein

In [ ]:
from joblib import Parallel, delayed
import numpy as np


def levenshtein_apply(levenshtein_flavor, rows):
    levenshtein = levenshtein_flavor()
    levenshtein_distance = np.vectorize(levenshtein.distance)
    
    distances = levenshtein_distance(rows[:, 0], rows[:, 1])
    
    return distances


def chunker(seq, size):
    return (seq[pos:pos + size] for pos in range(0, len(seq), size))


def levenshtein_batch(rows, levenshtein_flavor, n_jobs=8, batch=10000):
    distances = Parallel(n_jobs=n_jobs)(
        delayed(levenshtein_apply)(levenshtein_flavor, chunk) 
        for chunk in tqdm(
            chunker(rows[['name_1', 'name_2']].values, batch),
            total=rows.shape[0] // batch,
        )
    )

    return np.concatenate(distances)

In [ ]:
%%time

for dataset in (train, test):
    dataset["levenstein"] = levenshtein_batch(dataset, levenshtein_flavor=Levenshtein)

In [ ]:
%%time

for dataset in (train, test):
    dataset["norm_levenstein"] = levenshtein_batch(dataset, levenshtein_flavor=NormalizedLevenshtein)

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
plt.figure(figsize=(6,6))

train.boxplot("levenstein", by="is_duplicate", ax=plt.gca())
plt.suptitle("")
plt.title("Levenstein  distance", fontsize=12);

In [ ]:
plt.figure(figsize=(6,6))

train.boxplot("norm_levenstein", by="is_duplicate", ax=plt.gca())
plt.suptitle("")
plt.title("Normalized levenstein  distance", fontsize=12);

# ??????? ??????

In [ ]:
FTS = ["levenstein", "norm_levenstein"]
THR = 0.9

In [ ]:
split = StratifiedShuffleSplit(1, train_size=0.8, random_state=RS)
tridx, cvidx = list(split.split(train[FTS], train["is_duplicate"]))[0]

In [ ]:
%%time

model = LogisticRegression(class_weight="balanced")
model.fit(train.iloc[tridx][FTS].values,
          train.iloc[tridx]["is_duplicate"].values)

In [ ]:
%%time

tr_preds = model.predict_proba(train.iloc[tridx][FTS].values)
cv_preds = model.predict_proba(train.iloc[cvidx][FTS].values)

In [ ]:
tr_preds = (tr_preds[:, 1] > THR).astype(np.int)
cv_preds = (cv_preds[:, 1] > THR).astype(np.int)

In [ ]:
print(classification_report(train.iloc[tridx]["is_duplicate"].values, tr_preds))

In [ ]:
print(classification_report(train.iloc[cvidx]["is_duplicate"].values, cv_preds))

In [ ]:
print(f1_score(train.iloc[tridx]["is_duplicate"].values, tr_preds))
print(f1_score(train.iloc[cvidx]["is_duplicate"].values, cv_preds))

# ????????? ???????

In [ ]:
sample_sub = pd.read_csv(DATA_DIR.joinpath("sample_submission.csv"), index_col="pair_id")

In [ ]:
sample_sub

In [ ]:
sample_sub["is_duplicate"] = (model.predict_proba(test[FTS].values)[:, 1] > THR).astype(np.int)

In [ ]:
sample_sub.is_duplicate.value_counts()

In [ ]:
sample_sub.to_csv(
    DATA_DIR.joinpath("baseline_submission.csv.gz"), compression='infer')